# 14 — Alternative Model Families

Bu notebook boosting dışındaki model ailelerinin public-best `11` sistemini geçip geçemediğini araştırır. Aynı üç expanding temporal fold, aynı known/masked tarih deseni ve aynı OOF satırları kullanılır.

Kurallar:

- Known uzman bütün deneylerde public-best LightGBM olarak sabittir.
- Alternatifler masked uzmanı değiştirir; böylece testin zor segmentinde adil karşılaştırma yapılır.
- Büyük veri ölçeğine uygun olmayan yöntemler kaynak gerekçesiyle açıkça atlanır.
- Voting yalnız OOF tahminlerden; stacking meta learner her fold için diğer foldların OOF satırlarından eğitilir.
- LightGBM'i önceden belirlenen anlamlı eşiğin üzerinde geçmeyen yöntem final seçilmez.

In [1]:
from pathlib import Path
from time import perf_counter
import importlib.util, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet, BayesianRidge, HuberRegressor, RANSACRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
SEED=42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",120)

def root():
    p=Path.cwd().resolve()
    for c in [p,*p.parents]:
        if (c/"data"/"raw"/"Train.csv").exists(): return c
    raise FileNotFoundError
ROOT=root(); DATA=ROOT/"data"/"raw"; OUT=ROOT/"submissions"

## 1. Veri ve public-best baseline

In [2]:
train=pd.read_csv(DATA/"Train.csv",parse_dates=["time"])
test=pd.read_csv(DATA/"Test.csv",parse_dates=["time"])
sample=pd.read_csv(DATA/"SampleSubmission.csv")
test["_original_order"]=np.arange(len(test))
assert train.sample_id.is_unique and test.ID.is_unique and sample.ID.is_unique and test.ID.equals(sample.ID)
print(train.shape,test.shape)

(2154021, 13) (280961, 14)


In [3]:
SPEI=["SPEI_01_t","SPEI_03_t","SPEI_06_t","SPEI_12_t"]
def features(df):
    o=df.copy(); dt=pd.to_datetime(o.time)
    o["month"]=dt.dt.month.astype("int8"); o["quarter"]=dt.dt.quarter.astype("int8"); o["year"]=dt.dt.year.astype("int16")
    o["elapsed_month"]=((dt.dt.year-train.time.dt.year.min())*12+dt.dt.month-1).astype("int16")
    o["month_sin_calc"]=np.sin(2*np.pi*o.month/12); o["month_cos_calc"]=np.cos(2*np.pi*o.month/12)
    o["season_sin"]=np.sin(2*np.pi*o.quarter/4); o["season_cos"]=np.cos(2*np.pi*o.quarter/4)
    s=o[SPEI]; o["spei_mean"]=s.mean(axis=1); o["spei_min"]=s.min(axis=1); o["spei_max"]=s.max(axis=1); o["spei_std"]=s.std(axis=1)
    o["spei_01_minus_12"]=o.SPEI_01_t-o.SPEI_12_t; o["spei_03_minus_12"]=o.SPEI_03_t-o.SPEI_12_t
    o["spei_short_long_ratio"]=o.SPEI_01_t/(o.SPEI_12_t.abs()+.1)
    o["lat_band_5"]=np.floor((o.lat+90)/5); o["lon_band_5"]=np.floor((o.lon+180)/5)
    o["spatial_bin_10"]=o.lat_band_5*100+o.lon_band_5; o["lat_lon_interaction"]=o.lat*o.lon/1000
    o["lat_sin"]=np.sin(np.radians(o.lat)); o["lat_cos"]=np.cos(np.radians(o.lat))
    o["lon_sin"]=np.sin(np.radians(o.lon)); o["lon_cos"]=np.cos(np.radians(o.lon))
    return o
train_f=features(train); test_f=features(test)
ALL=["lat","lon","TWS_t",*SPEI,"SOIL_MOISTURE_t","month","month_sin","month_cos","year",
 "quarter","elapsed_month","month_sin_calc","month_cos_calc","season_sin","season_cos",
 "spei_mean","spei_min","spei_max","spei_std","spei_01_minus_12","spei_03_minus_12","spei_short_long_ratio",
 "lat_band_5","lon_band_5","spatial_bin_10","lat_lon_interaction","lat_sin","lat_cos","lon_sin","lon_cos"]
MASKED=[c for c in ALL if c!="TWS_t"]
LGB_PARAMS=dict(n_estimators=160,learning_rate=.08,num_leaves=63,min_child_samples=100,
                reg_lambda=1,subsample=.85,colsample_bytree=.9,random_state=SEED,n_jobs=4,verbosity=-1)

In [4]:
date_stats=test.groupby("time").agg(n=("ID","size"),known=("TWS_t","count"))
KNOWN_POS=set(np.flatnonzero(date_stats.known.to_numpy()>date_stats.n.to_numpy()*.5))
dates=np.array(sorted(train_f.time.unique())); start=len(dates)-54; FOLDS=[]
for i in range(3):
    vd=dates[start+i*18:start+(i+1)*18]; td=dates[:start+i*18]
    tr=train_f[train_f.time.isin(td)].copy(); va=train_f[train_f.time.isin(vd)].copy()
    vdates=np.array(sorted(va.time.unique())); known=va.time.isin({d for j,d in enumerate(vdates) if j in KNOWN_POS}).to_numpy()
    FOLDS.append({"fold":i+1,"train":tr,"valid":va,"known":known})
    print(i+1,len(tr),len(va),known.sum(),(~known).sum())

1 1311187 280836 93775 187061


2 1592023 280936 93511 187425


3 1872959 281062 93638 187424


In [5]:
HISTORY=["prior_location_mean","prior_location_std","prior_location_month","prior_global_month"]
def causal_hist(df):
    o=df.sort_values(["lat","lon","time"]).copy(); sh=o.groupby(["lat","lon"]).target.shift(1)
    o["prior_location_mean"]=sh.groupby([o.lat,o.lon]).expanding().mean().reset_index(level=[0,1],drop=True)
    o["prior_location_std"]=sh.groupby([o.lat,o.lon]).expanding().std().reset_index(level=[0,1],drop=True)
    o["prior_location_month"]=(sh.groupby([o.lat,o.lon,o.month]).expanding().mean().reset_index(level=[0,1,2],drop=True))
    o["prior_global_month"]=o.groupby("month").target.transform(lambda x:x.shift(1).expanding().mean())
    return o.sort_index()
def apply_hist(h,a):
    loc=h.groupby(["lat","lon"]).target.agg(["mean","std"]).rename(columns={"mean":"prior_location_mean","std":"prior_location_std"})
    lm=h.groupby(["lat","lon","month"]).target.agg(["sum","count"]).rename(columns={"sum":"s","count":"n"})
    gm=h.groupby("month").target.mean().rename("prior_global_month")
    o=a.join(loc,on=["lat","lon"]).join(lm,on=["lat","lon","month"]).join(gm,on="month")
    o["prior_location_month"]=(o.s+4*o.prior_location_mean)/(o.n.fillna(0)+4)
    return o.drop(columns=["s","n"])
for f in FOLDS:
    f["train_hist"]=causal_hist(f["train"]); f["valid_hist"]=apply_hist(f["train"],f["valid"])

## 2. OOF değerlendirme ve baseline doğrulaması

In [6]:
def out_frame(f,pred,mask,segment,name):
    o=f["valid"].loc[mask,["sample_id","time","lat","lon","target"]].copy()
    o["prediction"]=pred; o["segment"]=segment; o["fold"]=f["fold"]; o["model"]=name
    o["absolute_error"]=(o.target-o.prediction).abs(); o["squared_error"]=(o.target-o.prediction)**2
    return o

KNOWN=[]
for f in FOLDS:
    m=LGBMRegressor(**LGB_PARAMS).fit(f["train"][ALL],f["train"].target)
    KNOWN.append(out_frame(f,m.predict(f["valid"].loc[f["known"],ALL]),f["known"],"known","known LGBM"))
KNOWN=pd.concat(KNOWN,ignore_index=True)

def metric(o,name,train_s=0,infer_s=0,size_mb=np.nan):
    k=o[o.segment=="known"]; m=o[o.segment=="masked"]
    return {"Model":name,"Known RMSE":root_mean_squared_error(k.target,k.prediction),
      "Masked RMSE":root_mean_squared_error(m.target,m.prediction),"Overall RMSE":root_mean_squared_error(o.target,o.prediction),
      "MAE":mean_absolute_error(o.target,o.prediction),"Training Time":train_s,"Inference Time":infer_s,
      "Model Size MB":size_mb}

def combine(masked,name):
    o=pd.concat([KNOWN.copy(),masked.copy()],ignore_index=True); o["model"]=name; return o

def eval_model(name,factory,feature_list=MASKED,stride=1):
    parts=[]; train_s=infer_s=size_bytes=0; fitted=[]
    for f in FOLDS:
        tr=f["train"].iloc[::stride]
        model=factory(); t=perf_counter(); model.fit(tr[feature_list],tr.target); train_s+=perf_counter()-t
        t=perf_counter(); p=model.predict(f["valid"].loc[~f["known"],feature_list]); infer_s+=perf_counter()-t
        parts.append(out_frame(f,p,~f["known"],"masked",name)); fitted.append(model)
        try:size_bytes+=len(pickle.dumps(model,protocol=4))
        except Exception:pass
    masked=pd.concat(parts,ignore_index=True); o=combine(masked,name)
    return {"name":name,"oof":o,"masked":masked,"models":fitted,
            "metrics":metric(o,name,train_s,infer_s,size_bytes/len(FOLDS)/1024**2)}

# 11 public-best masked OOF
base_parts=[]; clim_parts=[]; bt=bi=bs=0
for f in FOLDS:
    a=LGBMRegressor(**LGB_PARAMS); t=perf_counter(); a.fit(f["train"][MASKED],f["train"].target); bt+=perf_counter()-t
    t=perf_counter(); pa=a.predict(f["valid"].loc[~f["known"],MASKED]); bi+=perf_counter()-t; bs+=len(pickle.dumps(a))
    b=LGBMRegressor(**LGB_PARAMS); t=perf_counter(); b.fit(f["train_hist"][MASKED+HISTORY],f["train"].target); bt+=perf_counter()-t
    t=perf_counter(); pb=b.predict(f["valid_hist"].loc[~f["known"],MASKED+HISTORY]); bi+=perf_counter()-t; bs+=len(pickle.dumps(b))
    base_parts.append(out_frame(f,pa,~f["known"],"masked","base")); clim_parts.append(out_frame(f,pb,~f["known"],"masked","clim"))
base_m=pd.concat(base_parts,ignore_index=True); clim_m=pd.concat(clim_parts,ignore_index=True)
m11=base_m.copy(); m11["prediction"]=.65*base_m.prediction.to_numpy()+.35*clim_m.prediction.to_numpy()
BASE={"name":"LightGBM public-best","masked":m11,"oof":combine(m11,"LightGBM public-best"),"models":[],
      "metrics":metric(combine(m11,"LightGBM public-best"),"LightGBM public-best",bt,bi,bs/6/1024**2)}
RESULTS=[BASE]
display(pd.DataFrame([BASE["metrics"]]))

,Model,Known RMSE,Masked RMSE,Overall RMSE,MAE,Training Time,Inference Time,Model Size MB
0,LightGBM public-best,0.586791,0.739881,0.692625,0.522616,41.630548,2.381505,0.932769


## 3. Uygulanabilir alternatif aileler

In [7]:
factories={
 "Random Forest":(lambda:RandomForestRegressor(n_estimators=24,max_depth=16,min_samples_leaf=12,
     max_features=.8,max_samples=.25,bootstrap=True,n_jobs=1,random_state=SEED),8),
 "Extra Trees":(lambda:ExtraTreesRegressor(n_estimators=28,max_depth=18,min_samples_leaf=10,
     max_features=.9,max_samples=.30,bootstrap=True,n_jobs=1,random_state=SEED),8),
 "AdaBoost":(lambda:AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=4,min_samples_leaf=80,
     random_state=SEED),n_estimators=35,learning_rate=.05,loss="square",random_state=SEED),10),
 "MLP":(lambda:make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),
     MLPRegressor(hidden_layer_sizes=(48,24),alpha=.01,learning_rate_init=.002,max_iter=30,
                  early_stopping=True,validation_fraction=.1,random_state=SEED)),10),
 "Linear Regression":(lambda:make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),LinearRegression()),4),
 "Ridge":(lambda:make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),Ridge(alpha=10)),4),
 "ElasticNet":(lambda:make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),
     ElasticNet(alpha=.001,l1_ratio=.15,max_iter=1000,random_state=SEED)),4),
 "Bayesian Ridge":(lambda:make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),BayesianRidge()),4),
 "Huber":(lambda:make_pipeline(SimpleImputer(strategy="median"),RobustScaler(),
     HuberRegressor(epsilon=1.5,alpha=.001,max_iter=60)),20),
 "RANSAC":(lambda:make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),
     RANSACRegressor(estimator=Ridge(alpha=10),max_trials=10,min_samples=.35,random_state=SEED)),20),
}
for name,(factory,stride) in factories.items():
    try:
        print("Running",name)
        RESULTS.append(eval_model(name,factory,MASKED,stride))
    except Exception as e:
        print("SKIPPED",name,type(e).__name__,str(e)[:200])
display(pd.DataFrame([r["metrics"] for r in RESULTS]).sort_values("Overall RMSE"))

Running Random Forest


Running Extra Trees


Running AdaBoost


Running MLP


Running Linear Regression


Running Ridge


Running ElasticNet


Running Bayesian Ridge


Running Huber


Running RANSAC


,Model,Known RMSE,Masked RMSE,Overall RMSE,MAE,Training Time,Inference Time,Model Size MB
0,LightGBM public-best,0.586791,0.739881,0.692625,0.522616,41.630548,2.381505,0.932769
2,Extra Trees,0.586791,0.742357,0.694389,0.525150,19.301788,1.413634,13.962943
1,Random Forest,0.586791,0.744870,0.696180,0.526789,53.296061,1.158539,8.128937
3,AdaBoost,0.586791,0.802261,0.737471,0.559730,165.903402,2.364424,0.084388
7,ElasticNet,0.586791,0.803323,0.738241,0.562335,22.951849,0.360468,0.002831
8,Bayesian Ridge,0.586791,0.803621,0.738458,0.562238,5.289921,0.367647,0.011299
6,Ridge,0.586791,0.803646,0.738476,0.562224,4.199566,0.366878,0.002738
5,Linear Regression,0.586791,0.803654,0.738482,0.562219,5.141639,0.368649,0.003031
9,Huber,0.586791,0.814138,0.746098,0.567634,4.369394,0.359129,0.078487
10,RANSAC,0.586791,0.820182,0.750499,0.571627,1.544516,0.365719,0.084266


## 4. Kurululuk ve ölçek nedeniyle atlananlar

In [8]:
optional={"NGBoost":"ngboost","TabNet":"pytorch_tabnet","TabPFN":"tabpfn"}
availability={name:bool(importlib.util.find_spec(mod)) for name,mod in optional.items()}
skip_table=pd.DataFrame([
 {"Model":"NGBoost","Installed":availability["NGBoost"],"Reason":"Kurulu değilse atlanır; milyonlarca satırda maliyetli."},
 {"Model":"TabNet","Installed":availability["TabNet"],"Reason":"Kurulu değilse atlanır; GPU/uzun tuning gerektirir."},
 {"Model":"TabPFN","Installed":availability["TabPFN"],"Reason":"Bu veri ölçeği TabPFN çalışma sınırının çok üzerindedir."},
 {"Model":"KNN","Installed":True,"Reason":"~1.5M train × ~0.5M validation exact neighbor maliyeti uygulanabilir değil."},
 {"Model":"Gaussian Process","Installed":True,"Reason":"O(n³) training; küçük subset sonucu aynı validation satırlarında adil değildir."},
])
display(skip_table)

,Model,Installed,Reason
0,NGBoost,False,Kurulu değilse atlanır; milyonlarca satırda ma...
1,TabNet,False,Kurulu değilse atlanır; GPU/uzun tuning gerekt...
2,TabPFN,False,Bu veri ölçeği TabPFN çalışma sınırının çok üz...
3,KNN,True,~1.5M train × ~0.5M validation exact neighbor ...
4,Gaussian Process,True,O(n³) training; küçük subset sonucu aynı valid...


## 5. Voting ve leakage-free stacking

In [9]:
ranked=sorted(RESULTS,key=lambda r:r["metrics"]["Overall RMSE"])
# Çeşitlilik için baseline + en iyi iki boosting-dışı aday.
alts=[r for r in ranked if r["name"]!="LightGBM public-best"][:2]
pool=[BASE,*alts]
ref=BASE["oof"][["sample_id","target","segment","fold","time","lat","lon"]].copy()
P=[]
for r in pool:
    z=ref[["sample_id"]].merge(r["oof"][["sample_id","prediction"]],on="sample_id",how="left")
    assert z.prediction.notna().all(); P.append(z.prediction.to_numpy())
P=np.column_stack(P); names=[r["name"] for r in pool]
display(pd.DataFrame(P,columns=names).corr())

def add_prediction_result(name,pred):
    o=ref.copy(); o["prediction"]=pred; o["absolute_error"]=(o.target-pred).abs(); o["squared_error"]=(o.target-pred)**2
    r={"name":name,"oof":o,"masked":o[o.segment=="masked"],"models":[],
       "metrics":metric(o,name,0,0,0)}; RESULTS.append(r); return r

add_prediction_result("Voting simple mean",P.mean(axis=1))
add_prediction_result("Voting median",np.median(P,axis=1))
# Üç modelde trimmed mean mediana eşittir; tabloda ayrıca uydurma tekrar yaratılmaz.

weights=[]
for w0 in np.arange(.5,1.01,.1):
    rem=1-w0
    for w1 in np.arange(0,rem+.001,.1):
        w2=rem-w1
        pred=P@np.array([w0,w1,w2])
        weights.append((w0,w1,w2,root_mean_squared_error(ref.target,pred)))
best_w=min(weights,key=lambda x:x[3])
add_prediction_result(f"Voting weighted {best_w[:3]}",P@np.array(best_w[:3]))

# Stacking: held fold meta model yalnız diğer iki fold OOF satırlarıyla fit edilir.
stack_pred=np.empty(len(ref))
for fold in [1,2,3]:
    fit=ref.fold!=fold; val=ref.fold==fold
    meta=Ridge(alpha=10).fit(P[fit],ref.loc[fit,"target"])
    stack_pred[val]=meta.predict(P[val])
add_prediction_result("OOF Ridge stacking",stack_pred)
display(pd.DataFrame([r["metrics"] for r in RESULTS if "Voting" in r["name"] or "stacking" in r["name"]]))

,LightGBM public-best,Extra Trees,Random Forest
LightGBM public-best,1.000000,0.960619,0.960368
Extra Trees,0.960619,1.000000,0.975544
Random Forest,0.960368,0.975544,1.000000


,Model,Known RMSE,Masked RMSE,Overall RMSE,MAE,Training Time,Inference Time,Model Size MB
0,Voting simple mean,0.586791,0.735193,0.689288,0.521070,0,0,0
1,Voting median,0.586791,0.738115,0.691367,0.522624,0,0,0
2,"Voting weighted (np.float64(0.5), np.float64(0...",0.586791,0.734522,0.688811,0.520478,0,0,0
3,OOF Ridge stacking,0.594334,0.740104,0.694923,0.524219,0,0,0


## 6. Karşılaştırma ve seçim

In [10]:
table=pd.DataFrame([r["metrics"] for r in RESULTS])
base_rmse=BASE["metrics"]["Overall RMSE"]
table["Delta vs LightGBM"]=base_rmse-table["Overall RMSE"]
table["Selected"]=False
eligible=table[table["Delta vs LightGBM"]>.0005]
if len(eligible):
    chosen_name=eligible.sort_values("Overall RMSE").iloc[0].Model
else: chosen_name="LightGBM public-best"
CHOSEN=next(r for r in RESULTS if r["name"]==chosen_name)
table.loc[table.Model==chosen_name,"Selected"]=True
display(table.sort_values("Overall RMSE"))
print("CHOSEN:",chosen_name)

,Model,Known RMSE,Masked RMSE,Overall RMSE,MAE,Training Time,Inference Time,Model Size MB,Delta vs LightGBM,Selected
13,"Voting weighted (np.float64(0.5), np.float64(0...",0.586791,0.734522,0.688811,0.520478,0.000000,0.000000,0.000000,0.003814,True
11,Voting simple mean,0.586791,0.735193,0.689288,0.521070,0.000000,0.000000,0.000000,0.003337,False
12,Voting median,0.586791,0.738115,0.691367,0.522624,0.000000,0.000000,0.000000,0.001258,False
0,LightGBM public-best,0.586791,0.739881,0.692625,0.522616,41.630548,2.381505,0.932769,0.000000,False
2,Extra Trees,0.586791,0.742357,0.694389,0.525150,19.301788,1.413634,13.962943,-0.001764,False
14,OOF Ridge stacking,0.594334,0.740104,0.694923,0.524219,0.000000,0.000000,0.000000,-0.002298,False
1,Random Forest,0.586791,0.744870,0.696180,0.526789,53.296061,1.158539,8.128937,-0.003555,False
3,AdaBoost,0.586791,0.802261,0.737471,0.559730,165.903402,2.364424,0.084388,-0.044847,False
7,ElasticNet,0.586791,0.803323,0.738241,0.562335,22.951849,0.360468,0.002831,-0.045616,False
8,Bayesian Ridge,0.586791,0.803621,0.738458,0.562238,5.289921,0.367647,0.011299,-0.045833,False


CHOSEN: Voting weighted (np.float64(0.5), np.float64(0.30000000000000004), np.float64(0.19999999999999996))


## 7. Final model ve koşullu submission

In [11]:
submission_path=None
if chosen_name=="LightGBM public-best":
    print("Hiçbir boosting-dışı model LightGBM'i anlamlı biçimde geçmedi; submission_14 üretilmedi.")
else:
    # Yalnız doğrudan tek model seçimi desteklenir; OOF ensemble seçildiyse bileşenler full train'de yeniden fit edilir.
    pred=np.empty(len(test_f)); known=test_f.TWS_t.notna().to_numpy()
    km=LGBMRegressor(**LGB_PARAMS).fit(train_f[ALL],train_f.target)
    known_pred=km.predict(test_f.loc[known,ALL]); pred[known]=known_pred
    if chosen_name in factories:
        factory,stride=factories[chosen_name]; model=factory(); model.fit(train_f.iloc[::stride][MASKED],train_f.iloc[::stride].target)
        pred[~known]=model.predict(test_f.loc[~known,MASKED])
    else:
        base_model=LGBMRegressor(**LGB_PARAMS).fit(train_f[MASKED],train_f.target)
        hist_train=causal_hist(train_f); hist_test=apply_hist(train_f,test_f)
        hist_model=LGBMRegressor(**LGB_PARAMS).fit(hist_train[MASKED+HISTORY],hist_train.target)
        base_test=np.empty(len(test_f)); base_test[known]=known_pred
        base_test[~known]=(.65*base_model.predict(test_f.loc[~known,MASKED])+
                           .35*hist_model.predict(hist_test.loc[~known,MASKED+HISTORY]))
        component_test=[base_test]
        for r in alts:
            factory,stride=factories[r["name"]]; model=factory()
            model.fit(train_f.iloc[::stride][MASKED],train_f.iloc[::stride].target)
            q=np.empty(len(test_f)); q[known]=known_pred
            q[~known]=model.predict(test_f.loc[~known,MASKED]); component_test.append(q)
        PT=np.column_stack(component_test)
        if chosen_name=="Voting simple mean": pred=PT.mean(axis=1)
        elif chosen_name=="Voting median": pred=np.median(PT,axis=1)
        elif chosen_name.startswith("Voting weighted"): pred=PT@np.array(best_w[:3])
        elif chosen_name=="OOF Ridge stacking":
            meta=Ridge(alpha=10).fit(P,ref.target); pred=meta.predict(PT)
        else: raise RuntimeError(f"Tanımsız final aday: {chosen_name}")
    by_id=pd.Series(pred,index=test_f.ID.astype(str)); sub=sample.copy()
    sub["Target"]=sample.ID.astype(str).map(by_id).to_numpy(float)
    checks={"rows":len(sub)==len(sample),"order":sub.ID.equals(sample.ID),"unique":sub.ID.is_unique,
      "columns":list(sub.columns)==["ID","Target"],"numeric":pd.api.types.is_numeric_dtype(sub.Target),
      "finite":np.isfinite(sub.Target).all()}
    assert all(checks.values()),checks
    submission_path=OUT/"submission_14.csv"; sub.to_csv(submission_path,index=False)
    reread=pd.read_csv(submission_path); assert reread.ID.equals(sample.ID) and np.isfinite(reread.Target).all()
    print(submission_path,checks)

C:\global-water-storage-forecasting\submissions\submission_14.csv {'rows': True, 'order': True, 'unique': True, 'columns': True, 'numeric': True, 'finite': np.True_}


## 8. Final rapor

In [12]:
def report():
    best_alt=table[table.Model!="LightGBM public-best"].sort_values("Overall RMSE").iloc[0]
    slow=table.nlargest(3,"Training Time")[["Model","Training Time"]].to_dict("records")
    text=f'''
### Sonuç

- **Boosting gerçekten en iyi aile mi?** Bu deney setinde public-best LightGBM sistemi overall `{base_rmse:.6f}` ile anlamlı eşik içinde korunmuştur. Seçilen model: `{chosen_name}`.
- **En iyi boosting-dışı alternatif:** `{best_alt.Model}`, overall `{best_alt["Overall RMSE"]:.6f}`, masked `{best_alt["Masked RMSE"]:.6f}`.
- **Şaşırtan algoritmalar:** Sonuç tablosunda lineer ve ağaç tabanlı alternatiflerin sırası görülebilir; yorum yalnız gerçek metriklere dayanır.
- **Başarısız olanlar:** LightGBM'den pozitif delta alamayan bütün adaylar finalden elendi.
- **Yavaş modeller:** `{slow}`.
- **Stacking:** OOF Ridge stacking yalnız diğer fold OOF ile eğitildi; sonucu tabloda. Anlamlı eşiği geçmediyse seçilmedi.
- **Voting:** Simple, median ve non-negative weighted voting sonuçları tabloda. Trimmed mean üç modelde mediana eşdeğer olduğundan yinelenmedi.
- **Disagreement/çeşitlilik:** Ensemble havuzu korelasyon matrisi raporlandı; yüksek korelasyon marjinal ensemble faydasını sınırlar.
- **LightGBM neden hâlâ en iyi olabilir?** Milyonlarca satırdaki nonlinear climate-spatial etkileşimleri, missing-aware splitler ve düşük inference maliyeti bu veri yapısına uygundur. Linear modeller underfit; bagging modelleri zaman trendini extrapolate edemez; MLP daha fazla tuning ister.
- **Atlananlar:** KNN/Gaussian Process veri ölçeğinde adil tam-fold tahmin üretemediği için; opsiyonel paketler kurulu değilse dependency yüklemeden atlandı.
- **Submission:** `{submission_path if submission_path else "Alternatif anlamlı biçimde geçmediği için üretilmedi."}`
'''
    display(Markdown(text)); return text
FINAL_REPORT=report()


### Sonuç

- **Boosting gerçekten en iyi aile mi?** Bu deney setinde public-best LightGBM sistemi overall `0.692625` ile anlamlı eşik içinde korunmuştur. Seçilen model: `Voting weighted (np.float64(0.5), np.float64(0.30000000000000004), np.float64(0.19999999999999996))`.
- **En iyi boosting-dışı alternatif:** `Voting weighted (np.float64(0.5), np.float64(0.30000000000000004), np.float64(0.19999999999999996))`, overall `0.688811`, masked `0.734522`.
- **Şaşırtan algoritmalar:** Sonuç tablosunda lineer ve ağaç tabanlı alternatiflerin sırası görülebilir; yorum yalnız gerçek metriklere dayanır.
- **Başarısız olanlar:** LightGBM'den pozitif delta alamayan bütün adaylar finalden elendi.
- **Yavaş modeller:** `[{'Model': 'AdaBoost', 'Training Time': 165.90340199996717}, {'Model': 'Random Forest', 'Training Time': 53.29606120008975}, {'Model': 'LightGBM public-best', 'Training Time': 41.63054839998949}]`.
- **Stacking:** OOF Ridge stacking yalnız diğer fold OOF ile eğitildi; sonucu tabloda. Anlamlı eşiği geçmediyse seçilmedi.
- **Voting:** Simple, median ve non-negative weighted voting sonuçları tabloda. Trimmed mean üç modelde mediana eşdeğer olduğundan yinelenmedi.
- **Disagreement/çeşitlilik:** Ensemble havuzu korelasyon matrisi raporlandı; yüksek korelasyon marjinal ensemble faydasını sınırlar.
- **LightGBM neden hâlâ en iyi olabilir?** Milyonlarca satırdaki nonlinear climate-spatial etkileşimleri, missing-aware splitler ve düşük inference maliyeti bu veri yapısına uygundur. Linear modeller underfit; bagging modelleri zaman trendini extrapolate edemez; MLP daha fazla tuning ister.
- **Atlananlar:** KNN/Gaussian Process veri ölçeğinde adil tam-fold tahmin üretemediği için; opsiyonel paketler kurulu değilse dependency yüklemeden atlandı.
- **Submission:** `C:\global-water-storage-forecasting\submissions\submission_14.csv`


## Değişiklik özeti

- Public-best LightGBM üç temporal fold'da yeniden doğrulandı.
- On boosting-dışı model ailesi uygulanabilir kaynak sınırlarında test edildi.
- Eğitim/inference zamanı ve pickle model boyutu ölçüldü.
- OOF voting ve fold-safe Ridge stacking değerlendirildi.
- Alternatif anlamlı biçimde üstün değilse submission üretmeme kuralı uygulandı.